# Сегментация листьев: baseline, improved baseline и собственная реализация U-Net

**Цель работы:** сравнить несколько архитектур сегментации на базовом пайплайне, улучшить бейзлайн с помощью гипотез и затем проверить, как техники улучшенного бейзлайна работают для самостоятельно реализованной модели.

В ноутбуке последовательно решаются две части задания:

1. **Улучшение бейзлайна**: выбор гипотез, проверка на валидации и сравнение лучших моделей с базовым решением.  
2. **Имплементация алгоритма**: собственная реализация U-Net, обучение в baseline- и improved-режимах и сравнение с библиотечными моделями.

**Основные метрики:** IoU, Dice и pixel accuracy. Для выбора лучшей модели используется в первую очередь **IoU**, так как для сегментации она лучше отражает качество совпадения маски предсказания с истинной маской.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not found")

In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
import zipfile
from pathlib import Path

ARCHIVE_PATH = Path("/content/archive.zip")
EXTRACT_PATH = Path("/content/dataset")

EXTRACT_PATH.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ARCHIVE_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Архив распакован в:", EXTRACT_PATH)

In [ ]:
!pip -q install -U kaggle segmentation-models-pytorch albumentations opencv-python-headless timm

In [ ]:
!find /content/dataset -maxdepth 5 | head -100

## Подготовка данных

Ниже определяется расположение исходных и аугментированных данных.  
Важный момент: для честного сравнения **train / val / test делятся только по исходным изображениям**, а аугментированные примеры добавляются только на этапе improved baseline. Это сделано, чтобы тест не загрязняется искусственно сгенерированными вариантами изображений.


In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/dataset")

BASE_DATA_ROOT = DATA_ROOT / "data" / "data"
AUG_DATA_ROOT = DATA_ROOT / "aug_data" / "aug_data"

print("BASE_DATA_ROOT:", BASE_DATA_ROOT)
print("AUG_DATA_ROOT :", AUG_DATA_ROOT)
print("BASE exists:", BASE_DATA_ROOT.exists())
print("AUG exists :", AUG_DATA_ROOT.exists())

In [ ]:
def collect_pairs_from_root(root: Path):
    pairs = []
    images_dir = root / "images"
    masks_dir = root / "masks"

    if not images_dir.exists() or not masks_dir.exists():
        raise FileNotFoundError(f"Не найдены папки images/masks в {root}")

    image_files = sorted(
        list(images_dir.glob("*.jpg")) +
        list(images_dir.glob("*.jpeg")) +
        list(images_dir.glob("*.png"))
    )

    for img_path in image_files:
        mask_path = masks_dir / f"{img_path.stem}.png"
        if mask_path.exists():
            pairs.append((img_path, mask_path))

    return pairs

baseline_pairs = collect_pairs_from_root(BASE_DATA_ROOT)
aug_pairs = collect_pairs_from_root(AUG_DATA_ROOT)

print("Baseline pairs:", len(baseline_pairs))
print("Augmented pairs:", len(aug_pairs))

for i in range(min(3, len(baseline_pairs))):
    print("IMG :", baseline_pairs[i][0])
    print("MASK:", baseline_pairs[i][1])
    print("-" * 40)

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

# Делим только оригинальные данные
train_pairs, test_pairs = train_test_split(
    baseline_pairs,
    test_size=0.15,
    random_state=SEED
)

train_pairs, val_pairs = train_test_split(
    train_pairs,
    test_size=0.1765,   # чтобы получить примерно 70/15/15
    random_state=SEED
)

print("Train:", len(train_pairs))
print("Val:", len(val_pairs))
print("Test:", len(test_pairs))

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

# Делим только оригинальные данные
train_pairs, test_pairs = train_test_split(
    baseline_pairs,
    test_size=0.15,
    random_state=SEED
)

train_pairs, val_pairs = train_test_split(
    train_pairs,
    test_size=0.1765,   # чтобы получить примерно 70/15/15
    random_state=SEED
)

print("Train:", len(train_pairs))
print("Val:", len(val_pairs))
print("Test:", len(test_pairs))

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def read_rgb(path):
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image


def read_mask(path):
    mask = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if mask is None:
        raise ValueError(f"Не удалось прочитать маску: {path}")
    if mask.ndim == 3:
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
    return mask

In [ ]:
import os
import random
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
import numpy as np

In [ ]:
SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 0

EPOCHS_BASELINE = 8
EPOCHS_IMPROVED = 10

LR = 1e-3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(SEED)

In [ ]:
class LeafSegDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        image = read_rgb(img_path)
        mask = read_mask(mask_path)

        # Бинаризация маски
        mask = (mask > 0).astype(np.uint8)

        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        mask = mask.unsqueeze(0).float()
        return image, mask

In [ ]:
baseline_train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])

baseline_eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])

In [ ]:
baseline_train_ds = LeafSegDataset(train_pairs, transform=baseline_train_transform)
baseline_val_ds = LeafSegDataset(val_pairs, transform=baseline_eval_transform)
baseline_test_ds = LeafSegDataset(test_pairs, transform=baseline_eval_transform)

baseline_train_loader = DataLoader(
    baseline_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

baseline_val_loader = DataLoader(
    baseline_val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

baseline_test_loader = DataLoader(
    baseline_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [ ]:
def compute_batch_metrics(logits, targets, threshold=0.5, eps=1e-7):
    """
    Вычисляет batch-level метрики сегментации.

    Args:
        logits: Логиты модели.
        targets: Истинные маски.
        threshold: Порог бинаризации.
        eps: Малое число для избежания деления на ноль.

    Returns:
        dict: IoU, Dice, Accuracy.
    """
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    tp = (preds * targets).sum(dim=(1, 2, 3))
    fp = (preds * (1 - targets)).sum(dim=(1, 2, 3))
    fn = ((1 - preds) * targets).sum(dim=(1, 2, 3))
    tn = ((1 - preds) * (1 - targets)).sum(dim=(1, 2, 3))

    iou = (tp + eps) / (tp + fp + fn + eps)
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    acc = (tp + tn + eps) / (tp + tn + fp + fn + eps)

    return {
        "iou": iou.mean().item(),
        "dice": dice.mean().item(),
        "acc": acc.mean().item(),
    }

In [ ]:
class BCEDiceLoss(nn.Module):
    """
    Комбинированная loss-функция BCE + Dice.
    """

    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode="binary", from_logits=True)

    def forward(self, logits, targets):
        return 0.5 * self.bce(logits, targets) + 0.5 * self.dice(logits, targets)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device=DEVICE):
    model.train()

    total_loss, total_iou, total_dice, total_acc = 0, 0, 0, 0
    total_batches = 0

    for images, masks in tqdm(loader, leave=False):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        metrics = compute_batch_metrics(logits.detach(), masks)

        total_loss += loss.item()
        total_iou += metrics["iou"]
        total_dice += metrics["dice"]
        total_acc += metrics["acc"]
        total_batches += 1

    return {
        "loss": total_loss / total_batches,
        "iou": total_iou / total_batches,
        "dice": total_dice / total_batches,
        "acc": total_acc / total_batches,
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device=DEVICE):
    """
    Оценивает модель на валидации или тесте.
    """
    model.eval()

    total_loss, total_iou, total_dice, total_acc = 0, 0, 0, 0
    total_batches = 0

    for images, masks in tqdm(loader, leave=False):
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        loss = criterion(logits, masks)

        metrics = compute_batch_metrics(logits, masks)

        total_loss += loss.item()
        total_iou += metrics["iou"]
        total_dice += metrics["dice"]
        total_acc += metrics["acc"]
        total_batches += 1

    return {
        "loss": total_loss / total_batches,
        "iou": total_iou / total_batches,
        "dice": total_dice / total_batches,
        "acc": total_acc / total_batches,
    }

In [ ]:
def build_smp_model(arch="Unet", encoder_name="resnet34", encoder_weights="imagenet"):
    model_class = getattr(smp, arch)
    model = model_class(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=1,
        activation=None
    )
    return model

In [ ]:
BASELINE_MODELS = [
    {"arch": "Unet", "encoder_name": "resnet34"},
    {"arch": "FPN", "encoder_name": "resnet34"},
    {"arch": "DeepLabV3Plus", "encoder_name": "resnet34"},
]

## Базовый бейзлайн

Для первого сравнения я взял три готовые архитектуры из `segmentation_models.pytorch`:

- **U-Net + ResNet34**
- **FPN + ResNet34**
- **DeepLabV3+ + ResNet34**

Это удобный старт, потому что модели устроены по-разному и интересно посмотреть, какая из них лучше сработает на задаче бинарной сегментации листьев.

**Комментарий к baseline:** на этом этапе я меняю только архитектуру модели. Все остальное остается одинаковым: размер изображений, функция потерь, оптимизатор, метрики и разбиение на train/val/test. За счет этого сравнение получается честным, и можно нормально понять, какая архитектура сама по себе работает лучше.

In [ ]:
os.makedirs("/content/checkpoints_baseline", exist_ok=True)

baseline_results = []
baseline_best_global = {"score": -1, "model_path": None, "config": None}

for cfg in BASELINE_MODELS:
    arch = cfg["arch"]
    encoder_name = cfg["encoder_name"]

    print("=" * 80)
    print(f"Baseline training: {arch} | {encoder_name}")

    model = build_smp_model(arch, encoder_name).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = BCEDiceLoss()

    best_val_iou = -1
    best_path = f"/content/checkpoints_baseline/{arch}_{encoder_name}.pth"

    for epoch in range(1, EPOCHS_BASELINE + 1):
        train_metrics = train_one_epoch(model, baseline_train_loader, optimizer, criterion)
        val_metrics = evaluate(model, baseline_val_loader, criterion)

        print(
            f"Epoch {epoch:02d}/{EPOCHS_BASELINE} | "
            f"train_loss={train_metrics['loss']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_iou={val_metrics['iou']:.4f} | "
            f"val_dice={val_metrics['dice']:.4f} | "
            f"val_acc={val_metrics['acc']:.4f}"
        )

        if val_metrics["iou"] > best_val_iou:
            best_val_iou = val_metrics["iou"]
            torch.save(model.state_dict(), best_path)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    test_metrics = evaluate(model, baseline_test_loader, criterion)

    row = {
        "arch": arch,
        "encoder": encoder_name,
        "best_val_iou": best_val_iou,
        "test_iou": test_metrics["iou"],
        "test_dice": test_metrics["dice"],
        "test_acc": test_metrics["acc"],
        "checkpoint": best_path,
    }
    baseline_results.append(row)

    if test_metrics["iou"] > baseline_best_global["score"]:
        baseline_best_global["score"] = test_metrics["iou"]
        baseline_best_global["model_path"] = best_path
        baseline_best_global["config"] = cfg

In [ ]:
baseline_results_df = pd.DataFrame(baseline_results).sort_values("test_iou", ascending=False).reset_index(drop=True)
display(baseline_results_df)

print("Best baseline model:")
print(baseline_best_global)

### Краткий анализ baseline-результатов

По итоговой таблице видно, что лучшим baseline-решением стала **FPN + ResNet34** с `test_iou ≈ 0.587`, `test_dice ≈ 0.712`, `test_acc ≈ 0.912`.

Это означает, что уже на базовом пайплайне FPN лучше остальных моделей использует многоуровневые признаки.  
При этом разрыв между моделями не очень большой, поэтому есть смысл проверять гипотезы по аугментациям, более сильным encoder'ам и более длинному обучению.


### Гипотезы для улучшения бейзлайна

После baseline видно, что модели дают близкие результаты, хотя **FPN + ResNet34** уже немного вырывается вперед.  
Это значит, что архитектура важна, но, скорее всего, заметный прирост можно получить не только за счет нее, а еще и за счет самого пайплайна обучения.

Поэтому дальше я хочу проверить несколько идей:

1. **Аугментации могут помочь модели лучше обобщать данные.**  
   Листья могут быть повернуты, отражены или немного сдвинуты, и модель должна уметь с этим справляться.

2. **Дополнительные аугментированные изображения могут сделать обучение устойчивее.**  
   За счет этого модель может меньше переобучаться и лучше работать на новых данных.

3. **`ResNet50` может оказаться сильнее, чем `ResNet34` в роли encoder'а.**  
   Но более сложная модель не всегда дает лучший результат, поэтому это нужно проверить на практике.

4. **Большее число эпох тоже может дать прирост качества.**  
   Возможно, в baseline модели просто не хватило времени, чтобы выйти на лучший результат.

In [ ]:
improved_train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=20,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.5
    ),
    A.ColorJitter(p=0.3),
    A.Normalize(),
    ToTensorV2(),
])

improved_eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])

In [ ]:
improved_train_pairs = train_pairs + aug_pairs

print("Improved train pairs:", len(improved_train_pairs))
print("Val pairs:", len(val_pairs))
print("Test pairs:", len(test_pairs))

In [ ]:
improved_train_ds = LeafSegDataset(improved_train_pairs, transform=improved_train_transform)
improved_val_ds = LeafSegDataset(val_pairs, transform=improved_eval_transform)
improved_test_ds = LeafSegDataset(test_pairs, transform=improved_eval_transform)

improved_train_loader = DataLoader(
    improved_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

improved_val_loader = DataLoader(
    improved_val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

improved_test_loader = DataLoader(
    improved_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [ ]:
IMPROVED_MODELS = [
    {"arch": "Unet", "encoder_name": "resnet34"},
    {"arch": "Unet", "encoder_name": "resnet50"},
    {"arch": "DeepLabV3Plus", "encoder_name": "resnet34"},
    {"arch": "DeepLabV3Plus", "encoder_name": "resnet50"},
]

In [ ]:
os.makedirs("/content/checkpoints_improved", exist_ok=True)

improved_results = []
improved_best_global = {"score": -1, "model_path": None, "config": None}

for cfg in IMPROVED_MODELS:
    arch = cfg["arch"]
    encoder_name = cfg["encoder_name"]

    print("=" * 80)
    print(f"Improved training: {arch} | {encoder_name}")

    model = build_smp_model(arch, encoder_name).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = BCEDiceLoss()

    best_val_iou = -1
    best_path = f"/content/checkpoints_improved/{arch}_{encoder_name}.pth"

    for epoch in range(1, EPOCHS_IMPROVED + 1):
        train_metrics = train_one_epoch(model, improved_train_loader, optimizer, criterion)
        val_metrics = evaluate(model, improved_val_loader, criterion)

        print(
            f"Epoch {epoch:02d}/{EPOCHS_IMPROVED} | "
            f"train_loss={train_metrics['loss']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_iou={val_metrics['iou']:.4f} | "
            f"val_dice={val_metrics['dice']:.4f} | "
            f"val_acc={val_metrics['acc']:.4f}"
        )

        if val_metrics["iou"] > best_val_iou:
            best_val_iou = val_metrics["iou"]
            torch.save(model.state_dict(), best_path)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    test_metrics = evaluate(model, improved_test_loader, criterion)

    row = {
        "arch": arch,
        "encoder": encoder_name,
        "best_val_iou": best_val_iou,
        "test_iou": test_metrics["iou"],
        "test_dice": test_metrics["dice"],
        "test_acc": test_metrics["acc"],
        "checkpoint": best_path,
    }
    improved_results.append(row)

    if test_metrics["iou"] > improved_best_global["score"]:
        improved_best_global["score"] = test_metrics["iou"]
        improved_best_global["model_path"] = best_path
        improved_best_global["config"] = cfg

In [ ]:
improved_results_df = pd.DataFrame(improved_results).sort_values("test_iou", ascending=False).reset_index(drop=True)
display(improved_results_df)

print("Best improved model:")
print(improved_best_global)

### Вывод по improved baseline

Лучшей моделью improved baseline стала **U-Net + ResNet34** с `test_iou ≈ 0.688`.  
Это заметно лучше лучшего baseline-результата (**FPN + ResNet34**, `test_iou ≈ 0.587`).

Следовательно, гипотеза о полезности улучшенного пайплайна **подтвердилась**: комбинация аугментаций, расширенной train-выборки и подбора моделей действительно дала прирост качества.

Отдельно стоит отметить, что **ResNet50 не стал лучшим encoder'ом**. Это полезный экспериментальный результат: более тяжелая модель не всегда лучше, особенно на сравнительно небольшом датасете.


In [ ]:
print("=== BASELINE RESULTS ===")
display(baseline_results_df)

print("=== IMPROVED RESULTS ===")
display(improved_results_df)

In [ ]:
def denormalize_image(tensor_img):
    """
    Возвращает изображение в диапазон [0, 1] после Normalize.

    Args:
        tensor_img: Tensor изображения.

    Returns:
        np.ndarray: Денормализованное изображение.
    """
    img = tensor_img.detach().cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = (img * std + mean).clip(0, 1)
    return img


@torch.no_grad()
def show_predictions(dataset, model, n=5, threshold=0.5):
    idxs = np.random.choice(len(dataset), min(n, len(dataset)), replace=False)
    plt.figure(figsize=(12, 4 * len(idxs)))

    for row, idx in enumerate(idxs, start=1):
        image, mask = dataset[idx]
        logits = model(image.unsqueeze(0).to(DEVICE))
        pred = (torch.sigmoid(logits)[0, 0] > threshold).float().cpu().numpy()

        image_vis = denormalize_image(image)
        mask_vis = mask[0].cpu().numpy()

        plt.subplot(len(idxs), 3, (row - 1) * 3 + 1)
        plt.imshow(image_vis)
        plt.title("Image")
        plt.axis("off")

        plt.subplot(len(idxs), 3, (row - 1) * 3 + 2)
        plt.imshow(mask_vis, cmap="gray")
        plt.title("Ground Truth")
        plt.axis("off")

        plt.subplot(len(idxs), 3, (row - 1) * 3 + 3)
        plt.imshow(pred, cmap="gray")
        plt.title("Prediction")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
best_cfg = improved_best_global["config"]

best_model = build_smp_model(
    arch=best_cfg["arch"],
    encoder_name=best_cfg["encoder_name"]
).to(DEVICE)

best_model.load_state_dict(torch.load(improved_best_global["model_path"], map_location=DEVICE))
best_model.eval()

show_predictions(improved_test_ds, best_model, n=5)

In [ ]:
baseline_results_df.to_csv("/content/baseline_results.csv", index=False)
improved_results_df.to_csv("/content/improved_results.csv", index=False)

print("Saved:")
print("/content/baseline_results.csv")
print("/content/improved_results.csv")

## Самостоятельная имплементация модели

В этой части я реализую собственную версию **U-Net**: отдельные блоки `DoubleConv`, `Down`, `Up` и итоговую сеть `CustomUNet`.

Дальше я обучу эту модель в двух вариантах и посмотрю, даст ли это прирост качества. Сначала проверю `CustomUNet` в базовом режиме, а затем — с дополнительными улучшениями, которые уже показали себя полезными в предыдущих экспериментах.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class DoubleConv(nn.Module):
    """
    Блок из двух сверток 3x3 с BatchNorm и ReLU.
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

In [ ]:
class Down(nn.Module):
    """
    Блок уменьшения размерности: MaxPool + DoubleConv.
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.pool_conv(x)

In [ ]:
class Up(nn.Module):
    """
    Блок увеличения размерности: ConvTranspose2d + skip connection + DoubleConv.
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(
            in_channels,
            in_channels // 2,
            kernel_size=2,
            stride=2
        )
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        diff_y = x2.size()[2] - x1.size()[2]
        diff_x = x2.size()[3] - x1.size()[3]

        x1 = F.pad(
            x1,
            [diff_x // 2, diff_x - diff_x // 2,
             diff_y // 2, diff_y - diff_y // 2]
        )

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

In [ ]:
class CustomUNet(nn.Module):
    """
    Реализация U-Net для бинарной сегментации.
    """
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)

        self.up1 = Up(1024, 512)
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)

        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        logits = self.outc(x)
        return logits

In [ ]:
custom_model = CustomUNet().to(DEVICE)
x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
y = custom_model(x)
print("Output shape:", y.shape)

In [ ]:
custom_baseline_model = CustomUNet().to(DEVICE)
criterion = BCEDiceLoss()
optimizer = torch.optim.Adam(custom_baseline_model.parameters(), lr=LR)

best_val_iou = -1
custom_baseline_path = "/content/custom_unet_baseline.pth"

custom_baseline_history = []

for epoch in range(1, EPOCHS_IMPROVED + 1):
    train_metrics = train_one_epoch(
        custom_baseline_model,
        baseline_train_loader,
        optimizer,
        criterion
    )
    val_metrics = evaluate(
        custom_baseline_model,
        baseline_val_loader,
        criterion
    )

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_iou": train_metrics["iou"],
        "train_dice": train_metrics["dice"],
        "train_acc": train_metrics["acc"],
        "val_loss": val_metrics["loss"],
        "val_iou": val_metrics["iou"],
        "val_dice": val_metrics["dice"],
        "val_acc": val_metrics["acc"],
    }
    custom_baseline_history.append(row)

    print(
        f"Epoch {epoch:02d}/{EPOCHS_IMPROVED} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_iou={val_metrics['iou']:.4f} | "
        f"val_dice={val_metrics['dice']:.4f} | "
        f"val_acc={val_metrics['acc']:.4f}"
    )

    if val_metrics["iou"] > best_val_iou:
        best_val_iou = val_metrics["iou"]
        torch.save(custom_baseline_model.state_dict(), custom_baseline_path)

In [ ]:
custom_baseline_model.load_state_dict(torch.load(custom_baseline_path, map_location=DEVICE))
custom_baseline_test_metrics = evaluate(custom_baseline_model, baseline_test_loader, criterion)

print("Custom U-Net baseline test metrics:")
print(custom_baseline_test_metrics)

In [ ]:
custom_baseline_results_df = pd.DataFrame([{
    "model": "CustomUNet",
    "mode": "baseline",
    "test_iou": custom_baseline_test_metrics["iou"],
    "test_dice": custom_baseline_test_metrics["dice"],
    "test_acc": custom_baseline_test_metrics["acc"],
    "checkpoint": custom_baseline_path
}])

display(custom_baseline_results_df)

### Краткий анализ собственной реализации в baseline-режиме

Собственная U-Net показывает `test_iou ≈ 0.479`, что ниже лучших библиотечных baseline-моделей.  
Это ожидаемо: библиотечные модели используют более сильные pretrained encoder'ы, тогда как реализованная вручную сеть проще и обучается с нуля.

Тем не менее результат остается рабочим, а значит реализация корректна и способна решать задачу сегментации.


In [ ]:
baseline_compare_df = baseline_results_df[["arch", "encoder", "test_iou", "test_dice", "test_acc"]].copy()
baseline_compare_df["model"] = baseline_compare_df["arch"] + "_" + baseline_compare_df["encoder"]

custom_row = pd.DataFrame([{
    "arch": "CustomUNet",
    "encoder": "-",
    "test_iou": custom_baseline_test_metrics["iou"],
    "test_dice": custom_baseline_test_metrics["dice"],
    "test_acc": custom_baseline_test_metrics["acc"],
    "model": "CustomUNet"
}])

baseline_compare_df = pd.concat([baseline_compare_df, custom_row], ignore_index=True)
baseline_compare_df = baseline_compare_df.sort_values("test_iou", ascending=False).reset_index(drop=True)

display(baseline_compare_df)

In [ ]:
custom_improved_model = CustomUNet().to(DEVICE)
criterion = BCEDiceLoss()
optimizer = torch.optim.Adam(custom_improved_model.parameters(), lr=LR)

best_val_iou = -1
custom_improved_path = "/content/custom_unet_improved.pth"

custom_improved_history = []

for epoch in range(1, EPOCHS_IMPROVED + 1):
    train_metrics = train_one_epoch(
        custom_improved_model,
        improved_train_loader,
        optimizer,
        criterion
    )
    val_metrics = evaluate(
        custom_improved_model,
        improved_val_loader,
        criterion
    )

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_iou": train_metrics["iou"],
        "train_dice": train_metrics["dice"],
        "train_acc": train_metrics["acc"],
        "val_loss": val_metrics["loss"],
        "val_iou": val_metrics["iou"],
        "val_dice": val_metrics["dice"],
        "val_acc": val_metrics["acc"],
    }
    custom_improved_history.append(row)

    print(
        f"Epoch {epoch:02d}/{EPOCHS_IMPROVED} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_iou={val_metrics['iou']:.4f} | "
        f"val_dice={val_metrics['dice']:.4f} | "
        f"val_acc={val_metrics['acc']:.4f}"
    )

    if val_metrics["iou"] > best_val_iou:
        best_val_iou = val_metrics["iou"]
        torch.save(custom_improved_model.state_dict(), custom_improved_path)

In [ ]:
custom_improved_model.load_state_dict(torch.load(custom_improved_path, map_location=DEVICE))
custom_improved_test_metrics = evaluate(custom_improved_model, improved_test_loader, criterion)

print("Custom U-Net improved test metrics:")
print(custom_improved_test_metrics)

### Что изменилось после добавления улучшений в `CustomUNet`

После добавления тех же идей к `CustomUNet` качество стало заметно лучше:

- `IoU`: примерно с **0.479** до **0.540**
- `Dice`: примерно с **0.614** до **0.677**
- `Accuracy`: примерно с **0.887** до **0.904**

То есть улучшения сработали не только на готовых библиотечных моделях, но и на моей собственной реализации.  
Это хороший знак: значит, дело не только в конкретной архитектуре, а еще и в самом процессе обучения.

При этом `CustomUNet` все равно остается слабее лучшей библиотечной модели.  
Скорее всего, это связано с тем, что библиотечные варианты используют более сильные pretrained encoder'ы, а моя реализация проще и обучается с нуля.

In [ ]:
custom_improved_results_df = pd.DataFrame([{
    "model": "CustomUNet",
    "mode": "improved",
    "test_iou": custom_improved_test_metrics["iou"],
    "test_dice": custom_improved_test_metrics["dice"],
    "test_acc": custom_improved_test_metrics["acc"],
    "checkpoint": custom_improved_path
}])

display(custom_improved_results_df)

In [ ]:
improved_compare_df = improved_results_df[["arch", "encoder", "test_iou", "test_dice", "test_acc"]].copy()
improved_compare_df["model"] = improved_compare_df["arch"] + "_" + improved_compare_df["encoder"]

custom_row_improved = pd.DataFrame([{
    "arch": "CustomUNet",
    "encoder": "-",
    "test_iou": custom_improved_test_metrics["iou"],
    "test_dice": custom_improved_test_metrics["dice"],
    "test_acc": custom_improved_test_metrics["acc"],
    "model": "CustomUNet"
}])

improved_compare_df = pd.concat([improved_compare_df, custom_row_improved], ignore_index=True)
improved_compare_df = improved_compare_df.sort_values("test_iou", ascending=False).reset_index(drop=True)

display(improved_compare_df)

In [ ]:
final_comparison_df = pd.DataFrame([
    {
        "group": "baseline_smp_best",
        "model": baseline_results_df.iloc[0]["arch"],
        "encoder": baseline_results_df.iloc[0]["encoder"],
        "iou": baseline_results_df.iloc[0]["test_iou"],
        "dice": baseline_results_df.iloc[0]["test_dice"],
        "acc": baseline_results_df.iloc[0]["test_acc"],
    },
    {
        "group": "improved_smp_best",
        "model": improved_results_df.iloc[0]["arch"],
        "encoder": improved_results_df.iloc[0]["encoder"],
        "iou": improved_results_df.iloc[0]["test_iou"],
        "dice": improved_results_df.iloc[0]["test_dice"],
        "acc": improved_results_df.iloc[0]["test_acc"],
    },
    {
        "group": "custom_baseline",
        "model": "CustomUNet",
        "encoder": "-",
        "iou": custom_baseline_test_metrics["iou"],
        "dice": custom_baseline_test_metrics["dice"],
        "acc": custom_baseline_test_metrics["acc"],
    },
    {
        "group": "custom_improved",
        "model": "CustomUNet",
        "encoder": "-",
        "iou": custom_improved_test_metrics["iou"],
        "dice": custom_improved_test_metrics["dice"],
        "acc": custom_improved_test_metrics["acc"],
    },
])

display(final_comparison_df.sort_values("iou", ascending=False).reset_index(drop=True))

## Итоговые выводы по работе

В целом работа показала, что даже простой стартовый набор моделей дает неплохую точку отсчета, но основной прирост качества получается не за счет “более тяжелой” модели сам по себе, а за счет нормального пайплайна обучения: аугментаций, расширения train-выборки и удачного выбора архитектуры.  
Среди baseline-моделей лучше всего себя показала `FPN + ResNet34`, а после улучшений лучшей стала `U-Net + ResNet34`.  
Собственная реализация `CustomUNet` ожидаемо уступила библиотечным моделям, но при этом работала корректно и тоже заметно улучшилась после применения тех же идей.  
Отдельно видно, что `ResNet50` на этом датасете не дал выигрыша, так что простое усложнение encoder'а здесь не оказалось самым полезным шагом.

In [ ]:
custom_baseline_model.eval()
show_predictions(baseline_test_ds, custom_baseline_model, n=5)

In [ ]:
custom_improved_model.eval()
show_predictions(improved_test_ds, custom_improved_model, n=5)

In [ ]:
custom_baseline_results_df.to_csv("/content/custom_baseline_results.csv", index=False)
custom_improved_results_df.to_csv("/content/custom_improved_results.csv", index=False)
final_comparison_df.to_csv("/content/final_comparison_results.csv", index=False)

print("Saved:")
print("/content/custom_baseline_results.csv")
print("/content/custom_improved_results.csv")
print("/content/final_comparison_results.csv")